# Lovecraft style LoRA on a Colab T4

Same method as `lora_small_model.ipynb` (style LoRA on the last N decoder layers, loss on
the assistant turn only). This version changes the data, the model and the hardware:

| | `lora_small_model.ipynb` | This notebook |
|---|---|---|
| Data | My messages, JSONL | H. P. Lovecraft stories, Project Gutenberg `.txt` files |
| Example | One message | One passage of about 250 words (packed paragraphs) |
| Train/eval split | Random over messages | Fixed held-out stories (no leakage, same eval set every run) |
| Model | `Qwen/Qwen3-8B`, 36 layers | `Qwen/Qwen3-1.7B`, 28 layers |
| GPU | 80 GB H100/A100, bf16 | Colab T4, 16 GB, **no bf16**: fp16 + fp32 LoRA (PART 3b) |
| Storage | VM disk + bucket | Google Drive (survives Colab disconnects) |

Plan:
  1. Colab setup: T4 runtime, install pinned libraries, mount Drive (PART 0)
  2. Put Gutenberg Lovecraft `.txt` files in Drive (PART 1b downloads them)
  3. Strip Gutenberg boilerplate, unwrap paragraphs, dedupe, pack into passages (PART 2)
  4. Hold out four fixed short stories for eval (PART 2)
  5. Chat format with loss masking; drop over-long examples instead of truncating (PART 3)
  6. Precision for a GPU without bf16: fp16 base weights, fp32 LoRA weights,
     fp16 autocast with loss scaling, fp32 fallback (PART 3b)
  7. LoRA on the last 8 of 28 layers, train, compare base vs adapted, save to Drive

Assumptions:
  - Colab runtime with a T4 (Runtime > Change runtime type > T4 GPU).
  - Books are Project Gutenberg plain-text UTF-8 files, one file per book or story.
  - The texts are public domain (Lovecraft died in 1937), so samples may be printed and
    outputs may be committed.

Limitations:
  - Fixed instruction ("Write a passage."): the adapter learns the unconditional style
    distribution. Prompts that ask for specific content are out of distribution.
  - Paragraph cleaning is heuristic. Headings, epigraphs and short dialogue lines under
    `min_paragraph_words` are dropped.
  - Qwen3 was trained in bf16. fp16 has a smaller range, so activations can overflow.
    PART 6 checks for this; the fallback is `force_fp32=True` (slower).
  - The 8B personal-style adapter cannot be loaded onto 1.7B (different width and depth).
    This is a fresh adapter.

## PART 0: Colab setup, imports and GPU check

Runtime > Change runtime type > T4 GPU (free). An L4 also works: the same code then
picks bf16 (PART 3b) and runs faster.

Run the install cell once per Colab session, then restart the session
(Runtime > Restart session) and continue from the Drive cell. The restart is needed
because Colab has already loaded an older `transformers`. Only the Hugging Face
libraries are installed, at the versions in `poetry.lock`. Colab's own CUDA build of
torch is kept. If transformers asks for a newer torch, install that version too.

When done: Runtime > Disconnect and delete runtime.

In [1]:
# Colab only. Locally, the poetry environment already has these
%pip install -q "transformers==5.19.0" "datasets==5.1.0" "peft==0.21.2" "accelerate==1.15.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [3]:
import hashlib
import json
import math
import os
import re
import time
import urllib.request
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
    set_seed,
)

/home/margot/projects/llm_xomnia_course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
assert torch.cuda.is_available(), (
    "No GPU found. Colab: Runtime > Change runtime type > T4 GPU."
)

props = torch.cuda.get_device_properties(0)
print(f"GPU:        {props.name}")
print(f"VRAM:       {props.total_memory / 1e9:.1f} GB")
print(f"Capability: {props.major}.{props.minor} (bf16 needs 8.0+)")
print(f"torch: {torch.__version__}, CUDA: {torch.version.cuda}")

AssertionError: No GPU found. Colab: Runtime > Change runtime type > T4 GPU.

## PART 1: Configuration and resource paths

Defaults point at Google Drive, so data, checkpoints and adapters survive a Colab
disconnect. Each one can be overridden with an environment variable
(`%env NAME=value` at the top of the notebook).

| Resource | Env var | Default | Notes |
|---|---|---|---|
| Work root | `LOVECRAFT_LORA_ROOT` | `/content/drive/MyDrive/lovecraft-lora` | Drive folder |
| Books | `LOVECRAFT_LORA_BOOKS` | `<root>/data/lovecraft` | Folder of Gutenberg `.txt` files, filled by PART 1b |
| Model cache | `LOVECRAFT_LORA_HF_CACHE` | `/content/hf_cache` | Local disk, faster than Drive. About 3.4 GB, downloaded again each session |
| Runs | `LOVECRAFT_LORA_RUNS` | `<root>/runs` | One folder per run: `checkpoints/`, `adapter/`, `run_config.json` |
| Previous adapter | `LOVECRAFT_LORA_INIT_ADAPTER` | unset | Continue from an earlier run's `adapter/` (Qwen3-1.7B only) |
| Run name | `LOVECRAFT_LORA_RUN_NAME` | generated | Set to an existing run name to resume it |
| Resume | `LOVECRAFT_LORA_RESUME` | `0` | `1` resumes the latest checkpoint of `LOVECRAFT_LORA_RUN_NAME` |

In [ ]:
# Must run before the config cell below: Paths reads the env vars when it runs.
# Defaults already point at Drive. Uncomment to resume after a Colab disconnect.
# %env LOVECRAFT_LORA_RUN_NAME=<run_name from the first run>
# %env LOVECRAFT_LORA_RESUME=1

In [ ]:
def env_path(name: str, default: Path) -> Path:
    """Read a path from an environment variable, else use the default."""
    return Path(os.environ.get(name, str(default)))


def env_optional_path(name: str) -> Path | None:
    """Read an optional path from an environment variable; None when unset."""
    value = os.environ.get(name, "").strip()
    return Path(value) if value else None


ROOT = env_path("LOVECRAFT_LORA_ROOT", Path("/content/drive/MyDrive/lovecraft-lora"))


@dataclass(frozen=True)
class Paths:
    """Resource locations on Colab; Drive unless overridden."""

    books_dir: Path = env_path("LOVECRAFT_LORA_BOOKS", ROOT / "data" / "lovecraft")
    hf_cache: Path = env_path("LOVECRAFT_LORA_HF_CACHE", Path("/content/hf_cache"))
    runs: Path = env_path("LOVECRAFT_LORA_RUNS", ROOT / "runs")
    init_adapter: Path | None = env_optional_path("LOVECRAFT_LORA_INIT_ADAPTER")


@dataclass(frozen=True)
class TrainConfig:
    """Model, data, precision, LoRA and optimization settings."""

    # Model and chat format
    model_name: str = "Qwen/Qwen3-1.7B"
    system_prompt: str = "You write prose in the style of H. P. Lovecraft."
    user_prompt: str = "Write a passage."  # fixed instruction for every example
    max_length: int = 512  # tokens, incl. chat template; longer examples are dropped
    seed: int = 42

    # Data: paragraphs packed into passages, whole books held out for eval
    min_paragraph_words: int = 8  # drops headings, titles, chapter numbers
    chunk_words: int = 250  # about 350 tokens of Lovecraft prose
    # Fixed held-out stories (file stems): short, varied, about 8% of the passages.
    # Fixed rather than random, so eval loss is comparable across runs and re-trains
    eval_books: tuple[str, ...] = ("cool_air", "festival", "pickmans_model", "temple")

    # Precision (PART 3b): fp16 on a T4; True loads and trains everything in fp32
    force_fp32: bool = False

    # LoRA: adapters only in the last n_last_layers decoder blocks (28 in total)
    n_last_layers: int = 8
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # Optimization, sized for a 16 GB T4
    n_epochs: int = 2
    learning_rate: float = 2e-4
    per_device_batch_size: int = 4
    grad_accum_steps: int = 4
    warmup_fraction: float = 0.05
    weight_decay: float = 0.0
    gradient_checkpointing: bool = False  # set True if you hit OOM
    eval_steps: int = 25  # about 130 optimizer steps in total
    save_steps: int = 25
    logging_steps: int = 5


paths = Paths()
cfg = TrainConfig()
set_seed(cfg.seed)

RESUME = os.environ.get("LOVECRAFT_LORA_RESUME", "0") == "1"
if RESUME and not os.environ.get("LOVECRAFT_LORA_RUN_NAME"):
    raise ValueError("LOVECRAFT_LORA_RESUME=1 needs LOVECRAFT_LORA_RUN_NAME.")

# Reuse an existing run name to resume it; otherwise a new versioned name
RUN_NAME = os.environ.get("LOVECRAFT_LORA_RUN_NAME") or (
    f"lovecraft-{cfg.model_name.split('/')[-1].lower()}"
    f"-last{cfg.n_last_layers}-r{cfg.lora_r}"
    f"-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
)

run_dir = paths.runs / RUN_NAME
checkpoint_dir = run_dir / "checkpoints"
adapter_dir = run_dir / "adapter"

# Prompts for PART 8. The first is the training instruction; the others test whether
# the style carries over to requests with content.
EVAL_PROMPTS = [
    cfg.user_prompt,
    "Write a passage about an old house at the edge of a New England town.",
    "Write a diary entry about a sound coming from beneath the cellar floor.",
]

for name, value in asdict(paths).items():
    print(f"{name:<14} {value}")
print(f"{'run_dir':<14} {run_dir}")
print(f"{'resume':<14} {RESUME}")

## PART 1b: Download the books

Plain-text files from Project Gutenberg, saved to Drive. Files already there are kept, so
re-running this cell after a disconnect downloads nothing.

The list is Lovecraft's own fiction on Gutenberg (author page:
gutenberg.org/ebooks/author/34724), about 200k words. It leaves out collaborations
(with Bishop, Heald, Price, Whitehead), because they are partly someone else's prose, and
the essays. All are public domain in the US (Gutenberg's check) and in the EU
(Lovecraft died in 1937).

The file name is the book name used for the train/eval split and in `run_config.json`.
Any extra `.txt` file you put in the folder is used as well.

In [ ]:
# Project Gutenberg ebook id -> file name. Lovecraft solo fiction only
GUTENBERG_BOOKS = {
    68283: "call_of_cthulhu",
    70652: "at_the_mountains_of_madness",
    50133: "dunwich_horror",
    31469: "shunned_house",
    68236: "colour_out_of_space",
    79842: "temple",
    73181: "shadow_over_innsmouth",
    73547: "case_of_charles_dexter_ward",
    73233: "haunter_of_the_dark",
    79401: "pickmans_model",
    68553: "festival",
    79405: "picture_in_the_house",
    73230: "thing_on_the_doorstep",
    72966: "horror_at_red_hook",
    70478: "silver_key",
    70486: "lurking_fear",
    68547: "he",
    73177: "cool_air",
    73182: "quest_of_iranon",
}
GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/{id}/pg{id}.txt"


def download_books(books: dict[int, str], books_dir: Path) -> list[Path]:
    """Download missing Gutenberg texts into books_dir; existing files are kept."""
    books_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for ebook_id, name in books.items():
        path = books_dir / f"{name}.txt"
        if not path.exists():
            urllib.request.urlretrieve(GUTENBERG_URL.format(id=ebook_id), path)
            time.sleep(1)  # be polite: Gutenberg blocks aggressive crawling
        files.append(path)
    return files


for f in download_books(GUTENBERG_BOOKS, paths.books_dir):
    print(f"{f.name:<35} {f.stat().st_size / 1e3:>8.0f} KB")

## PART 2: Load, clean and split the books

Validate before spending GPU time. Steps per file:
  1. Keep only the text between the Gutenberg `*** START OF ...` and `*** END OF ...` markers
  2. Split on blank lines, unwrap hard line breaks, remove `_italics_` markers
  3. Drop short paragraphs (headings) and Gutenberg notes (`[Illustration]`, `Produced by`)

Then over all files: drop duplicate paragraphs (overlapping anthologies), and pack
consecutive paragraphs of each book into passages of at most `chunk_words`. Paragraphs
longer than that are split at sentence ends.

In [ ]:
TEXT_COLUMN = "text"
BOOK_COLUMN = "book"
MIN_RECOMMENDED_WORDS = 100_000

GUTENBERG_START = re.compile(
    r"\*\*\*\s*START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^*]*\*\*\*", re.IGNORECASE
)
GUTENBERG_END = re.compile(
    r"\*\*\*\s*END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK", re.IGNORECASE
)
BOILERPLATE = re.compile(
    r"^(?:\[|\*|produced by|transcriber|e-?text prepared|this e-?book)", re.IGNORECASE
)
FOOTNOTE_REF = re.compile(r"\[\d+\]")  # inline markers like "[1]"
SENTENCE_END = re.compile(r"(?<=[.!?])\s+(?=[\"'“‘A-Z])")


def read_book(path: Path) -> str:
    """Read one Gutenberg text and keep only the body between its markers."""
    text = path.read_text(encoding="utf-8-sig", errors="replace").replace("\r\n", "\n")
    start, end = GUTENBERG_START.search(text), GUTENBERG_END.search(text)
    return text[start.end() if start else 0 : end.start() if end else len(text)]


def split_paragraphs(body: str, min_words: int) -> list[str]:
    """Unwrap hard line breaks; drop headings, short lines and Gutenberg notes."""
    paragraphs = (
        re.sub(r"\s+", " ", FOOTNOTE_REF.sub("", block).replace("_", "")).strip()
        for block in re.split(r"\n\s*\n", body)
    )
    return [
        p for p in paragraphs
        if len(p.split()) >= min_words and not BOILERPLATE.match(p)
    ]


def load_paragraphs(books_dir: Path, min_words: int) -> pd.DataFrame:
    """One row per paragraph, in reading order; book is the file stem."""
    files = sorted(books_dir.glob("*.txt"))
    if not files:
        raise FileNotFoundError(
            f"No .txt files in {books_dir}. Run PART 1b or set LOVECRAFT_LORA_BOOKS."
        )
    rows = [
        {BOOK_COLUMN: f.stem, TEXT_COLUMN: p}
        for f in files
        for p in split_paragraphs(read_book(f), min_words)
    ]
    return pd.DataFrame(rows, columns=[BOOK_COLUMN, TEXT_COLUMN])


def paragraph_units(paragraph: str, max_words: int) -> list[str]:
    """The paragraph itself, or its sentences when it is longer than max_words."""
    if len(paragraph.split()) <= max_words:
        return [paragraph]
    return SENTENCE_END.split(paragraph)


def pack_chunks(paragraphs: list[str], max_words: int) -> list[str]:
    """Greedily pack consecutive paragraphs into passages of at most max_words."""
    chunks: list[str] = []
    current, n_words = "", 0
    for paragraph in paragraphs:
        for i, unit in enumerate(paragraph_units(paragraph, max_words)):
            unit_words = len(unit.split())
            if current and n_words + unit_words > max_words:
                chunks.append(current)
                current, n_words = "", 0
            separator = "\n\n" if i == 0 else " "  # keep paragraph breaks
            current = f"{current}{separator}{unit}" if current else unit
            n_words += unit_words
    if current:
        chunks.append(current)
    return chunks


def build_passages(paragraphs_df: pd.DataFrame, max_words: int) -> pd.DataFrame:
    """Chunk each book separately so no passage spans two books."""
    rows = [
        {BOOK_COLUMN: book, TEXT_COLUMN: chunk}
        for book, group in paragraphs_df.groupby(BOOK_COLUMN, sort=True)
        for chunk in pack_chunks(group[TEXT_COLUMN].tolist(), max_words)
    ]
    return pd.DataFrame(rows, columns=[BOOK_COLUMN, TEXT_COLUMN])


raw_paragraphs = load_paragraphs(paths.books_dir, cfg.min_paragraph_words)
paragraphs_df = raw_paragraphs.drop_duplicates(subset=TEXT_COLUMN)  # keeps first
passages_df = build_passages(paragraphs_df, cfg.chunk_words)
n_words = passages_df[TEXT_COLUMN].str.split().str.len()

print(f"Books:                 {passages_df[BOOK_COLUMN].nunique()}")
print(f"Paragraphs:            {len(raw_paragraphs):,} "
      f"(duplicates dropped: {len(raw_paragraphs) - len(paragraphs_df):,})")
print(f"Passages:              {len(passages_df):,}")
print(f"Total words:           {n_words.sum():,}")
print(f"Words p50 / p90 / max: {n_words.median():.0f} / "
      f"{n_words.quantile(0.9):.0f} / {n_words.max()}")
if n_words.sum() < MIN_RECOMMENDED_WORDS:
    print(f"WARNING: fewer than {MIN_RECOMMENDED_WORDS:,} words, expect overfitting.")

passages_df.groupby(BOOK_COLUMN)[TEXT_COLUMN].agg(
    passages="count", words=lambda s: s.str.split().str.len().sum()
)

In [ ]:
# Spot-check the cleaning: no headers, licence text or line-wrapping artefacts
for passage in passages_df.sample(3, random_state=cfg.seed)[TEXT_COLUMN]:
    print(passage[:600], "\n", "-" * 80)

In [ ]:
def split_by_book(
    df: pd.DataFrame, eval_books: tuple[str, ...]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Hold out whole books, so eval passages never share a story with training."""
    missing = set(eval_books) - set(df[BOOK_COLUMN])
    if missing:
        raise ValueError(f"Eval books not found in {paths.books_dir}: {sorted(missing)}")
    is_eval = df[BOOK_COLUMN].isin(eval_books)
    if is_eval.all():
        raise ValueError("Every book is an eval book: nothing left to train on.")
    return df[~is_eval], df[is_eval]


train_df, eval_df = split_by_book(passages_df, cfg.eval_books)
train_books = sorted(train_df[BOOK_COLUMN].unique())
eval_books = sorted(eval_df[BOOK_COLUMN].unique())
print(f"Eval share of passages: {len(eval_df) / len(passages_df):.1%}")

split = DatasetDict({
    "train": Dataset.from_pandas(train_df[[TEXT_COLUMN]], preserve_index=False),
    "test": Dataset.from_pandas(eval_df[[TEXT_COLUMN]], preserve_index=False),
})
print(f"Eval books: {eval_books}")
print(split)

## PART 3: Chat formatting and loss masking

Each passage becomes one conversation:

```
system:    <style instruction>          masked (label = -100)
user:      Write a passage.             masked
assistant: <Lovecraft passage><end>     trained
```

The model's own chat template is used, so the adapter works with normal chat inference.
Qwen3 thinking mode is disabled. Examples longer than `max_length` are dropped, not
truncated: truncation would cut the end-of-turn token and teach the model not to stop.

In [ ]:
IGNORE_INDEX = -100  # label value skipped by the cross-entropy loss

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, cache_dir=paths.hf_cache)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_messages(
    cfg: TrainConfig, instruction: str, passage: str | None = None
) -> list[dict[str, str]]:
    """Build system/user(/assistant) turns; the assistant turn is the passage."""
    turns = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user", "content": instruction},
    ]
    if passage is not None:
        turns.append({"role": "assistant", "content": passage})
    return turns


def render_chat(
    tokenizer: PreTrainedTokenizerBase,
    turns: list[dict[str, str]],
    add_generation_prompt: bool,
) -> str:
    """Render turns with the model's chat template, thinking disabled."""
    # enable_thinking is a Qwen3 template variable; other templates ignore it
    return tokenizer.apply_chat_template(
        turns,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    )


def tokenize_example(
    row: dict[str, str],
    tokenizer: PreTrainedTokenizerBase,
    cfg: TrainConfig,
    prompt_ids: list[int],
) -> dict[str, list[int]]:
    """Tokenize one passage and mask every token except the passage itself."""
    full_text = render_chat(
        tokenizer,
        build_messages(cfg, cfg.user_prompt, row[TEXT_COLUMN]),
        add_generation_prompt=False,
    )
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError(
            "Chat template mismatch: the prompt is not a prefix of the full "
            "conversation. Check the template after swapping models."
        )
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


# The prompt is identical for every example: tokenize it once
PROMPT_IDS = tokenizer(
    render_chat(
        tokenizer, build_messages(cfg, cfg.user_prompt), add_generation_prompt=True
    ),
    add_special_tokens=False,
)["input_ids"]

In [ ]:
tokenized = split.map(
    tokenize_example,
    fn_kwargs={"tokenizer": tokenizer, "cfg": cfg, "prompt_ids": PROMPT_IDS},
    remove_columns=[TEXT_COLUMN],
    desc="Tokenizing",
)
n_before = {name: len(ds) for name, ds in tokenized.items()}
tokenized = tokenized.filter(lambda row: len(row["input_ids"]) <= cfg.max_length)
train_ds, eval_ds = tokenized["train"], tokenized["test"]

lengths = np.array([len(ids) for ids in train_ds["input_ids"]])
print(f"Train examples: {len(train_ds):,} "
      f"(dropped as too long: {n_before['train'] - len(train_ds)})")
print(f"Eval examples:  {len(eval_ds):,} "
      f"(dropped as too long: {n_before['test'] - len(eval_ds)})")
print(f"Tokens per example: p50={np.percentile(lengths, 50):.0f}, "
      f"p90={np.percentile(lengths, 90):.0f}, max={lengths.max()}")
print(f"Train tokens per epoch: {lengths.sum():,}")

In [ ]:
# The loss should cover only the passage and the end-of-turn token
example = train_ds[0]
print("FULL INPUT:")
print(tokenizer.decode(example["input_ids"]))
print()
print("TRAINED ON:")
print(tokenizer.decode([t for t in example["labels"] if t != IGNORE_INDEX]))

## PART 3b: Precision on a GPU without bf16 (T4)

The T4 has compute capability 7.5. bf16 needs 8.0+ (Ampere). Do not rely on
`torch.cuda.is_bf16_supported()`: by default it also counts software emulation and can
return True on a T4. The capability is checked instead.

fp16 setup on a T4:
  - **Frozen base weights in fp16**: about 3.4 GB for Qwen3-1.7B, and the T4 tensor cores
    run fp16 matmuls fast.
  - **Trainable LoRA weights in fp32**. Two reasons: the fp16 `GradScaler` refuses to
    unscale fp16 gradients ("Attempting to unscale FP16 gradients"), and Adam updates at
    lr 2e-4 underflow when stored in fp16. PEFT usually upcasts adapters already;
    `upcast_trainable_params` makes this explicit and PART 5 asserts it.
  - **`fp16=True` in `TrainingArguments`**: fp16 autocast plus dynamic loss scaling. The
    scaler skips steps with inf/nan gradients and lowers the scale. A few skipped steps
    at the start are normal.

Overflow risk: Qwen3 was trained in bf16 (max about 3e38). fp16 tops out at 65504, so
large activations can become inf. PART 6 stops if the baseline eval loss is not finite.
If it fails, or train loss turns nan, either switch the runtime to an L4 (bf16), or set
`force_fp32=True`: fp32 weights (about 6.9 GB), no autocast, several times slower, but
numerically safe.

On an A100/H100 the same code picks bf16 automatically.

In [ ]:
def pick_precision(force_fp32: bool) -> tuple[torch.dtype, bool, bool]:
    """Return (weight dtype, bf16 flag, fp16 flag) for the GPU in use."""
    if force_fp32:
        return torch.float32, False, False
    if torch.cuda.get_device_capability(0) >= (8, 0):  # Ampere+: native bf16
        return torch.bfloat16, True, False
    return torch.float16, False, True


def upcast_trainable_params(model: torch.nn.Module) -> int:
    """Cast trainable (LoRA) parameters to fp32; frozen weights keep their dtype."""
    n_cast = 0
    for param in model.parameters():
        if param.requires_grad and param.dtype != torch.float32:
            param.data = param.data.float()
            n_cast += 1
    return n_cast


MODEL_DTYPE, USE_BF16, USE_FP16 = pick_precision(cfg.force_fp32)
print(f"Base weights: {MODEL_DTYPE}")
print(f"Autocast:     {'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'none (fp32)'}")
print("LoRA weights: torch.float32")

## PART 4: Load the base model

Weights in the PART 3b dtype, whole model on GPU 0. SDPA attention: FlashAttention 2
does not support the T4. The number of decoder layers is read from the config.

In [ ]:
model_config = AutoConfig.from_pretrained(cfg.model_name, cache_dir=paths.hf_cache)
n_layers = model_config.num_hidden_layers
last_layer_ids = list(range(n_layers - cfg.n_last_layers, n_layers))

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    dtype=MODEL_DTYPE,
    attn_implementation="sdpa",
    device_map={"": 0},
    cache_dir=paths.hf_cache,
)
base_model.config.use_cache = False  # no KV cache needed during training

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Parameters:        {n_params / 1e9:.2f} B")
print(f"Decoder layers:    {n_layers}")
print(f"Layers to adapt:   {last_layer_ids}")
print(f"GPU memory (load): {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## PART 5: LoRA on the last N layers

Same as the 8B notebook: `layers_to_transform` restricts LoRA to the last
`n_last_layers` blocks; `target_modules` covers attention and MLP projections. No layer
before the first adapter has trainable parameters, so backward runs through 8 of 28 layers.

Expected for Qwen3-1.7B (hidden 2048, MLP 6144, 8 KV heads), last 8 layers, r=16, all
7 projections: 622,592 per layer, about 5.0M trainable parameters (about 0.3%).

Re-train mode (`LOVECRAFT_LORA_INIT_ADAPTER` set): the previous adapter is loaded as
trainable and its saved LoRA settings apply. `n_last_layers` and `lora_*` are ignored.

In [ ]:
def build_lora_config(cfg: TrainConfig, layer_ids: list[int]) -> LoraConfig:
    """LoRA config restricted to the given decoder layer indices."""
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        layers_to_transform=layer_ids,
        layers_pattern="layers",  # module path is model.layers.<i>.<...>
        bias="none",
    )


def adapted_layer_ids(model: torch.nn.Module) -> list[int]:
    """Return the decoder layer indices that hold trainable parameters."""
    ids = {
        int(name.split(".layers.")[1].split(".")[0])
        for name, param in model.named_parameters()
        if param.requires_grad and ".layers." in name
    }
    return sorted(ids)


if paths.init_adapter is not None:
    # Re-train: continue from a previous run's adapter
    model = PeftModel.from_pretrained(
        base_model, str(paths.init_adapter), is_trainable=True
    )
else:
    model = get_peft_model(base_model, build_lora_config(cfg, last_layer_ids))

n_cast = upcast_trainable_params(model)
trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
assert trainable_dtypes == {torch.float32}, f"LoRA weights not fp32: {trainable_dtypes}"

model.print_trainable_parameters()
adapted_layers = adapted_layer_ids(model)
print(f"Upcast to fp32:       {n_cast} tensors (0 means PEFT already did it)")
print(f"Layers with adapters: {adapted_layers}")
if paths.init_adapter is None:
    assert adapted_layers == last_layer_ids, "LoRA landed on unexpected layers"

## PART 6: Train

Hugging Face `Trainer` handles gradient accumulation, autocast and loss scaling (fp16 on
the T4), the cosine schedule with warmup, gradient clipping, checkpoints and
best-checkpoint selection on eval loss. Checkpoints go to Drive every `save_steps`, so a
Colab disconnect costs at most that many steps (resume: PART 10).

Metric: eval loss (and perplexity) on held-out books, passage tokens only.
Baseline: the same loss before training (the base model, since a fresh adapter has B = 0).

In [ ]:
effective_batch = cfg.per_device_batch_size * cfg.grad_accum_steps
total_steps = math.ceil(len(train_ds) / effective_batch) * cfg.n_epochs
warmup_steps = max(1, round(cfg.warmup_fraction * total_steps))

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    num_train_epochs=cfg.n_epochs,
    per_device_train_batch_size=cfg.per_device_batch_size,
    per_device_eval_batch_size=cfg.per_device_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=cfg.weight_decay,
    max_grad_norm=1.0,
    bf16=USE_BF16,
    fp16=USE_FP16,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    eval_steps=cfg.eval_steps,
    save_strategy="steps",
    save_steps=cfg.save_steps,
    save_total_limit=2,  # Drive space
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=cfg.logging_steps,
    report_to="none",  # "mlflow" or "wandb" once tracking is set up
    seed=cfg.seed,
    data_seed=cfg.seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        padding=True,
        label_pad_token_id=IGNORE_INDEX,
        pad_to_multiple_of=8,
    ),
    processing_class=tokenizer,
)

print(f"Effective batch size: {effective_batch}")
print(f"Optimizer steps:      {total_steps} (warmup {warmup_steps})")

In [ ]:
baseline_metrics = trainer.evaluate()
assert math.isfinite(baseline_metrics["eval_loss"]), (
    "Baseline eval loss is not finite: fp16 overflow. Set force_fp32=True (PART 3b)."
)
print(f"Baseline eval loss: {baseline_metrics['eval_loss']:.4f} "
      f"(perplexity {math.exp(baseline_metrics['eval_loss']):.1f})")

In [ ]:
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=RESUME or None)
final_metrics = trainer.evaluate()  # best checkpoint is loaded at the end

nan_steps = [
    h["step"] for h in trainer.state.log_history
    if "loss" in h and not math.isfinite(h["loss"])
]
if nan_steps:
    print(f"WARNING: non-finite train loss at steps {nan_steps}. "
          "Set force_fp32=True (PART 3b).")

print(f"Train runtime:      {train_result.metrics['train_runtime'] / 60:.1f} min")
print(f"Peak GPU memory:    {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(f"Baseline eval loss: {baseline_metrics['eval_loss']:.4f}")
print(f"Final eval loss:    {final_metrics['eval_loss']:.4f} "
      f"(perplexity {math.exp(final_metrics['eval_loss']):.1f})")

## PART 7: Training curves

Eval loss is on held-out books, so it measures style transfer to unseen stories.
Eval loss rising while train loss keeps falling means memorizing the training stories:
fewer epochs, lower `lora_r`, or more books.

In [ ]:
history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(*zip(*train_points), label="Train loss")
ax.plot(*zip(*eval_points), marker="o", label="Eval loss (held-out books)")
ax.axhline(baseline_metrics["eval_loss"], color="gray", linestyle="--",
           label="Baseline eval loss")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Loss (passage tokens only)")
ax.set_title("Lovecraft style LoRA training")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## PART 8: Base vs adapted output

Same model object: `disable_adapter()` switches LoRA off for the base output.
Same seed for both, so differences come from the adapter, not sampling noise.
Generation runs in the PART 3b dtype.

In [ ]:
def generate_reply(
    model: PeftModel,
    tokenizer: PreTrainedTokenizerBase,
    cfg: TrainConfig,
    instruction: str,
    max_new_tokens: int = 200,
) -> str:
    """Generate one sampled passage for an instruction in the training format."""
    text = render_chat(
        tokenizer, build_messages(cfg, instruction), add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    inputs = inputs.to(model.device)
    set_seed(cfg.seed)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


model.eval()
for instruction in EVAL_PROMPTS:
    with model.disable_adapter():
        base_reply = generate_reply(model, tokenizer, cfg, instruction)
    adapted_reply = generate_reply(model, tokenizer, cfg, instruction)
    print(f"PROMPT:  {instruction}")
    print(f"BASE:    {base_reply}")
    print(f"ADAPTED: {adapted_reply}")
    print()

## PART 9: Save the adapter and run record

Only the adapter is saved (about 20 MB), not the 3.4 GB base model. `run_config.json`
records config, the hash of the book files, the book split, the actual LoRA config
(also correct in re-train mode), precision and metrics.

In [ ]:
def hash_files(files: list[Path]) -> str:
    """SHA256 over file names and contents, in sorted order."""
    digest = hashlib.sha256()
    for f in sorted(files):
        digest.update(f.name.encode())
        digest.update(f.read_bytes())
    return digest.hexdigest()


run_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(adapter_dir))  # adapter weights and tokenizer only

run_record = {
    "run_name": RUN_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "books_sha256": hash_files(list(paths.books_dir.glob("*.txt"))),
    "train_books": train_books,
    "eval_books": eval_books,
    "paths": asdict(paths),
    "config": asdict(cfg),
    "lora_config": model.peft_config["default"].to_dict(),
    "adapted_layers": adapted_layers,
    "gpu": torch.cuda.get_device_name(0),
    "model_dtype": str(MODEL_DTYPE),
    "autocast": "bf16" if USE_BF16 else "fp16" if USE_FP16 else "none",
    "n_train": len(train_ds),
    "n_eval": len(eval_ds),
    "baseline_eval_loss": baseline_metrics["eval_loss"],
    "final_eval_loss": final_metrics["eval_loss"],
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
}
(run_dir / "run_config.json").write_text(
    json.dumps(run_record, indent=2, default=str)
)

print(f"Saved run to {run_dir}")
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size / 1e6:>8.2f} MB")

## PART 10: Re-training and next iterations

| Goal | Set | Then |
|---|---|---|
| Continue after a Colab disconnect | `LOVECRAFT_LORA_RUN_NAME=<run_name>`, `LOVECRAFT_LORA_RESUME=1` | Mount Drive, run all |
| Train further on more books | `LOVECRAFT_LORA_INIT_ADAPTER=<root>/runs/<run_name>/adapter` | Run all. A new run folder is created |
| fp16 overflow (nan loss) | `force_fp32=True` in `TrainConfig` | Restart session, run all |
| Out of memory | `gradient_checkpointing=True` or `per_device_batch_size=2`, `grad_accum_steps=8` | Restart session, run all |

Re-trains keep the same `eval_books`, so eval loss stays comparable and no eval story
leaks into training. Do not change `eval_books` between a run and its re-train: the
previous adapter has seen the stories it trained on.

Next iterations (not in this notebook):
  - Varied instructions (topic or opening sentence per passage), so content prompts
    are in distribution
  - Ablate `n_last_layers` (4, 8, 16, 28) and `lora_r` against held-out-book eval loss
  - Style metrics on generations (sentence length, type-token ratio, archaic word rate)
    vs held-out books, with bootstrap confidence intervals
  - Blind test: real vs generated passages
  - Same pipeline on my messages with Qwen3-1.7B on the T4, to compare the two styles